# Marine Seismic Reflection Survey — Bass Strait

Synthetic shot gather for a towed hydrophone streamer survey using Kennett
reflectivity with free surface reverberations, source/receiver ghosts, and
Bessel-weighted wavenumber summation.

**Configuration**: `configs/example_survey.yml`  
**Module**: `cubic_scattering.seismic_survey`

In [ ]:
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Add project root to path so cubic_scattering is importable
sys.path.insert(0, str(Path("..").resolve()))

from cubic_scattering import (
    compute_shot_gather,
    load_survey_config,
)

logging.basicConfig(level=logging.INFO)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

## 1. Load configuration

In [ ]:
config_path = Path("../configs/example_survey.yml")
stack, survey, gather_config = load_survey_config(config_path)

print(f"Water depth:      {survey.water_depth} m")
print(f"Source depth:     {survey.source_depth} m")
print(f"Receiver depth:   {survey.receiver_depth} m")
print(f"Receiver type:    {survey.receiver_type}")
print(
    f"Offsets:          {survey.offsets[0]:.0f} – {survey.offsets[-1]:.0f} m  ({len(survey.offsets)} receivers)"
)
print(f"Frequency band:   {gather_config.f_min} – {gather_config.f_max} Hz")
print(f"Ricker peak:      {gather_config.f_peak} Hz")
print(f"Record length:    {gather_config.T} s")
print(f"Free surface:     {gather_config.free_surface}")
print(f"\nLayer stack ({len(stack.layers)} layers):")
for i, lay in enumerate(stack.layers):
    h = f"{lay.thickness:.0f} m" if np.isfinite(lay.thickness) else "half-space"
    beta_str = f"{lay.beta:.0f}" if lay.beta > 0 else "  0 (fluid)"
    print(f"  {i}: α={lay.alpha:.0f}  β={beta_str}  ρ={lay.rho:.0f}  h={h}")

## 2. Compute shot gather

In [ ]:
%%time
result = compute_shot_gather(stack, survey, gather_config)

## 3. Plot shot gather (variable-area wiggle + image)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)

t_max = 3.0  # s — display window
t_mask = result.time <= t_max
t = result.time[t_mask]
g = result.gather[:, t_mask]

# --- Left: image plot ---
clip = 0.05 * np.abs(g).max()
ax = axes[0]
ax.imshow(
    g.T,
    aspect="auto",
    extent=[result.offsets[0], result.offsets[-1], t[-1], t[0]],
    cmap="gray",
    vmin=-clip,
    vmax=clip,
    interpolation="bilinear",
)
ax.set_xlabel("Offset (m)")
ax.set_ylabel("Time (s)")
ax.set_title("Image plot")

# --- Right: wiggle traces ---
ax = axes[1]
# Plot every nth trace to avoid clutter
n_traces = 40
trace_stride = max(1, len(result.offsets) // n_traces)
trace_indices = np.arange(0, len(result.offsets), trace_stride)

offsets_sel = result.offsets[trace_indices]
traces = g[trace_indices, :]

# Normalise each trace
scale = 0.8 * np.median(np.diff(offsets_sel)) / (np.abs(traces).max() + 1e-30)

for i, idx in enumerate(trace_indices):
    x0 = result.offsets[idx]
    tr = traces[i] * scale
    ax.plot(x0 + tr, t, "k-", linewidth=0.4)
    ax.fill_betweenx(t, x0, x0 + tr, where=(tr > 0), color="k", alpha=0.6, linewidth=0)

ax.set_xlim(result.offsets[0] - 50, result.offsets[-1] + 50)
ax.set_xlabel("Offset (m)")
ax.set_title("Wiggle traces")

fig.suptitle(
    f"Bass Strait — {survey.receiver_type}, z_s={survey.source_depth} m, "
    f"z_r={survey.receiver_depth} m, water={survey.water_depth} m",
    fontsize=12,
)
fig.tight_layout()
plt.show()

## 4. Reflectivity spectrum

In [ ]:
dw = 2.0 * np.pi / gather_config.T
nwm = gather_config.nw - 1
freq = np.arange(1, nwm + 1) * dw / (2.0 * np.pi)  # Hz

# p_max
alpha_min = min(lay.alpha for lay in stack.layers)
p_max = 1.2 / alpha_min
dp = p_max / gather_config.np_slow
p_samples = np.arange(1, gather_config.np_slow + 1) * dp

fig, ax = plt.subplots(figsize=(10, 6))
R_abs = np.abs(result.reflectivity)
clip_r = np.percentile(R_abs, 99)
ax.imshow(
    R_abs.T,
    aspect="auto",
    extent=[p_samples[0], p_samples[-1], freq[-1], freq[0]],
    cmap="hot",
    vmin=0,
    vmax=clip_r,
    interpolation="bilinear",
)
ax.set_xlabel("Slowness (s/m)")
ax.set_ylabel("Frequency (Hz)")
ax.set_title("$|R^{\\downarrow\\downarrow}_{PP}(p, \\omega)|$ — Sub-ocean reflectivity")
fig.colorbar(ax.images[0], ax=ax, label="|R|")
fig.tight_layout()
plt.show()

## 5. Single trace at near and far offset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, offset_target in zip(axes, [200.0, 2000.0], strict=True):
    idx = np.argmin(np.abs(result.offsets - offset_target))
    actual_offset = result.offsets[idx]
    tr = result.gather[idx, t_mask]

    ax.plot(tr, t, "k-", linewidth=0.6)
    ax.fill_betweenx(t, 0, tr, where=(tr > 0), color="steelblue", alpha=0.4)
    ax.fill_betweenx(t, 0, tr, where=(tr < 0), color="firebrick", alpha=0.4)
    ax.set_xlabel("Amplitude")
    ax.set_title(f"Offset = {actual_offset:.0f} m")
    ax.axvline(0, color="gray", linewidth=0.5, linestyle="--")

axes[0].set_ylabel("Time (s)")
axes[0].invert_yaxis()
fig.suptitle("Near-offset vs far-offset traces", fontsize=12)
fig.tight_layout()
plt.show()

## 6. Compare with and without free surface reverberations

In [ ]:
from dataclasses import replace

gather_no_fs = replace(gather_config, free_surface=False)
result_no_fs = compute_shot_gather(stack, survey, gather_no_fs)

fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)

g_fs = result.gather[:, t_mask]
g_no = result_no_fs.gather[:, t_mask]
clip = 0.05 * max(np.abs(g_fs).max(), np.abs(g_no).max())

for ax, data, title in zip(
    axes,
    [g_fs, g_no],
    ["With free surface reverberations", "Without (primaries only)"],
    strict=True,
):
    ax.imshow(
        data.T,
        aspect="auto",
        extent=[result.offsets[0], result.offsets[-1], t[-1], t[0]],
        cmap="gray",
        vmin=-clip,
        vmax=clip,
        interpolation="bilinear",
    )
    ax.set_xlabel("Offset (m)")
    ax.set_title(title)

axes[0].set_ylabel("Time (s)")
fig.suptitle("Effect of water-column multiples", fontsize=12)
fig.tight_layout()
plt.show()

## 7. Amplitude vs offset (AVO) at the water-bottom reflection

In [ ]:
# Pick the water-bottom reflection time for each offset
# Two-way time: t(x) = sqrt(t0^2 + (x/alpha_water)^2)
t0_wb = 2 * survey.water_depth / survey.water_alpha  # normal-incidence TWT

# Use the no-free-surface gather for cleaner AVO
g_avo = result_no_fs.gather[:, t_mask]

# Extract peak amplitude in a window around the expected arrival
half_win = 0.02  # s
avo_amp = np.zeros(len(result.offsets))

for i, x in enumerate(result.offsets):
    t_arr = np.sqrt(t0_wb**2 + (x / survey.water_alpha) ** 2)
    mask_win = (t >= t_arr - half_win) & (t <= t_arr + half_win)
    if mask_win.any():
        avo_amp[i] = np.max(np.abs(g_avo[i, mask_win]))

# Convert offset to angle (approximate: sin(theta) = p * alpha)
theta_deg = np.degrees(
    np.arcsin(
        np.clip(
            result.offsets
            / np.sqrt(result.offsets**2 + (2 * survey.water_depth) ** 2)
            / survey.water_alpha
            * survey.water_alpha,
            0,
            1,
        )
    )
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(result.offsets, avo_amp, "k-", linewidth=1)
axes[0].set_xlabel("Offset (m)")
axes[0].set_ylabel("Peak amplitude")
axes[0].set_title("AVO — water-bottom reflection")
axes[0].grid(True, alpha=0.3)

# Normalised
axes[1].plot(result.offsets, avo_amp / (avo_amp[0] + 1e-30), "k-", linewidth=1)
axes[1].set_xlabel("Offset (m)")
axes[1].set_ylabel("Normalised amplitude")
axes[1].set_title("AVO (normalised to near offset)")
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 8. Source wavelet and ghost spectrum

In [ ]:
from cubic_scattering import receiver_ghost, ricker_source_spectrum, source_ghost

freq_plot = np.linspace(1, 100, 500)
omega_plot = 2 * np.pi * freq_plot

S = ricker_source_spectrum(omega_plot, gather_config.f_peak)

# Ghost at normal incidence (p=0)
p_zero = np.array([0.0])
sg = source_ghost(omega_plot, p_zero, survey.source_depth, survey.water_alpha)
rg = receiver_ghost(
    omega_plot, p_zero, survey.receiver_depth, survey.water_alpha, survey.receiver_type
)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))

axes[0, 0].plot(freq_plot, np.abs(S), "k-")
axes[0, 0].set_title(f"Ricker wavelet (f_peak = {gather_config.f_peak} Hz)")
axes[0, 0].set_ylabel("|S(f)|")

axes[0, 1].plot(
    freq_plot,
    np.abs(sg[0, :]),
    "b-",
    label=f"Source ghost (z_s={survey.source_depth} m)",
)
axes[0, 1].plot(
    freq_plot,
    np.abs(rg[0, :]),
    "r-",
    label=f"Receiver ghost (z_r={survey.receiver_depth} m)",
)
axes[0, 1].set_title("Ghost operators (normal incidence)")
axes[0, 1].set_ylabel("|Ghost|")
axes[0, 1].legend(fontsize=8)

# Combined source + ghosts
combined = np.abs(S) * np.abs(sg[0, :]) * np.abs(rg[0, :])
axes[1, 0].plot(freq_plot, combined, "k-")
axes[1, 0].set_title("Combined: source $\\times$ ghosts")
axes[1, 0].set_xlabel("Frequency (Hz)")
axes[1, 0].set_ylabel("|S $\\cdot$ G_s $\\cdot$ G_r|")

# Ghost notch frequencies
f_notch_s = survey.water_alpha / (2 * survey.source_depth)
f_notch_r = survey.water_alpha / (2 * survey.receiver_depth)
axes[1, 1].plot(freq_plot, 20 * np.log10(combined / combined.max() + 1e-10), "k-")
axes[1, 1].axvline(
    f_notch_s,
    color="b",
    linestyle="--",
    alpha=0.7,
    label=f"Source notch {f_notch_s:.0f} Hz",
)
axes[1, 1].axvline(
    f_notch_r,
    color="r",
    linestyle="--",
    alpha=0.7,
    label=f"Receiver notch {f_notch_r:.0f} Hz",
)
axes[1, 1].set_title("Combined spectrum (dB)")
axes[1, 1].set_xlabel("Frequency (Hz)")
axes[1, 1].set_ylabel("dB")
axes[1, 1].set_ylim(-40, 5)
axes[1, 1].legend(fontsize=8)

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()